In [ ]:
# ================= NOTEBOOK 2: XLM-ROBERTA FINE-TUNING BASELINE =================
!pip install -q transformers datasets accelerate scikit-learn

import torch
import numpy as np
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score
from scipy.special import softmax

# Ensure train, val, test are loaded from Notebook 1
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")

MODEL_ID = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def tokenize_func(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

# Convert pandas to HF Dataset
from datasets import Dataset
train_ds = Dataset.from_pandas(train[["text", "y"]].rename(columns={"y": "labels"}))
val_ds   = Dataset.from_pandas(val[["text", "y"]].rename(columns={"y": "labels"}))
test_ds  = Dataset.from_pandas(test[["text", "y"]].rename(columns={"y": "labels"}))

train_tok = train_ds.map(tokenize_func, batched=True)
val_tok   = val_ds.map(tokenize_func, batched=True)
test_tok  = test_ds.map(tokenize_func, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_ID, num_labels=2).to(device)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, preds, average="macro")
    acc = accuracy_score(labels, preds)
    return {"macro_f1": macro_f1, "accuracy": acc}

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/xlmr_out",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=2e-5,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=True,
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    compute_metrics=compute_metrics,
)

print("Starting XLM-R Training...")
trainer.train()

# Final Test Evaluation
out = trainer.predict(test_tok)
preds = np.argmax(out.predictions, axis=-1)
probs = softmax(out.predictions, axis=-1)[:, 1]

test_f1 = f1_score(test["y"], preds, average="macro")
test_acc = accuracy_score(test["y"], preds)
test_auroc = roc_auc_score(test["y"], probs)

print(f"\n--- XLM-R TEST RESULTS ---")
print(f"Macro-F1 : {test_f1:.4f}")
print(f"Accuracy : {test_acc:.4f}")
print(f"AUROC    : {test_auroc:.4f}")